### <I style="color: yellow;">!! Note that this notebook was ran in google colab so it may have some <span style="color: red;"> path </span> and <span style="color: red;"> dependencies issues </span> !!</I>

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="hItSI9SouxLMfpSmjJ3X")
project = rf.workspace("ultima-itso6").project("player-detection-q7zza")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Player-detection-1 in yolov8:: 100%|██████████| 1757/1757 [00:00<00:00, 5868.30it/s]


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.4 MB/s eta 0:00:00


In [11]:
import os

# The main folder name
base_path = 'Player-detection-1'

# List of subfolders that contain 'labels'
splits = ['train', 'test', 'valid']

def convert_to_single_class(base_dir, split_list):
    count = 0
    for split in split_list:
        # Construct path to the labels folder (e.g., Player-detection-1/train/labels)
        label_folder = os.path.join(base_dir, split, 'labels')

        if not os.path.exists(label_folder):
            print(f"Skipping: {label_folder} (not found)")
            continue

        for filename in os.listdir(label_folder):
            if filename.endswith('.txt'):
                file_path = os.path.join(label_folder, filename)

                with open(file_path, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        # Force Class ID to 0 regardless of what it was
                        parts[0] = '0'
                        new_lines.append(" ".join(parts) + "\n")

                with open(file_path, 'w') as f:
                    f.writelines(new_lines)
                count += 1

    print(f"Finished! Processed {count} label files across {split_list}.")

convert_to_single_class(base_path, splits)

Finished! Processed 876 label files across ['train', 'test', 'valid'].


In [4]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
INPUT_VIDEO = "/content/drive/MyDrive/padel_game_model_training/input_video_shortest.mp4"

In [ ]:
yolo = YOLO("yolov8m.pt")

results = yolo.train(
    data = "Player-detection-1/data.yaml",
    epochs = 80,
    patience = 20,
    batch=16,
    project = "/content/drive/MyDrive/padel_game_model_training",
    name = "player_detection",
    save = True
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Player-detection-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=player_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [8]:
model = YOLO("yolov8m.pt")

result = model.track(INPUT_VIDEO, conf = 0.4, iou = 0.4, tracker="bytetrack.yaml", save = True, persist=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/371) /content/drive/MyDrive/padel_game_model_training/input_video_shortest.mp4: 384x640 6 persons, 1 sports ball, 120.4ms
video 1/1 (frame 2/371) /content/drive/MyDrive/padel_game_model_training/input_video_shortest.mp4: 384x640 6 persons, 1 sports ball, 26.1ms
video 1/1 (frame 3/371) /content/drive/MyDrive/padel_game_model_training/input_video_shortest.mp4: 384x640 6 persons, 25.5ms
video 1/1 (frame 4/371) /content/drive/MyDrive/padel